# 从零实现 Transformer 核心机制

让我们像 2017 年的研究者一样，带着问题、公式和草稿纸，从零开始发明 Transformer 的核心机制。我们将一步步追问：为什么要自注意力？为什么是点积？为什么要除以 √d_k ？每一处都会先给出思路，再写出数学表达式，最后用最基础的张量操作实现并观察输出，绝不使用 `nn.MultiheadAttention` 这样的成品工具。

---

## 1. 从 RNN 的枷锁到"让每个词直接看到所有词"

**Idea**
循环神经网络（RNN）必须按顺序读句子：第 t 步的隐藏状态必须等第 t-1 步算完。这带来了两个痛苦：
- 无法并行训练。
- 长距离依赖被多次乘法削弱。

我们希望每个位置的表示能**直接与序列中所有位置交互**，并把那些最相关位置的信息"汇总"过来。一种自然的想法：用内积来衡量两个词向量的相关性，相关性越高，分配越多的注意力。

**Mathematical expression**
假设我们有序列的表征矩阵 $X \in \mathbb{R}^{n \times d}$，其中 n 为序列长度，d 为每个词的向量维度。我们定义一个相关性矩阵：
$$
\text{scores} = X X^\top \quad (\text{大小 } n \times n)
$$
矩阵的第 i 行第 j 列就是词 i 与词 j 的内积相似度。随后用 softmax 得到归一化的注意力权重，并用它去加权求和 X 自身：
$$
\text{Attention}(X) = \text{softmax}(X X^\top) X
$$
但目前这个版本有一个致命问题。

**Code & Output**
我们先用 PyTorch 试试看：

In [1]:
import torch
import torch.nn.functional as F

# 假设序列长度 n=4，每个词向量维度 d=64
torch.manual_seed(42)
n, d = 4, 64
X = torch.randn(n, d)  # 随机初始化一个序列表征

# 计算未缩放的点积分数
scores = X @ X.T        # @ 表示矩阵乘法，形状 [4,4]
print("未缩放分数矩阵:\n", scores)
# 观察：某些分数绝对值已经到 20~30，softmax 后会极其尖锐

未缩放分数矩阵:
 tensor([[70.5184, 16.1441, 11.8250, -2.4180],
        [16.1441, 47.7958,  8.3251,  8.7938],
        [11.8250,  8.3251, 72.2133,  2.2382],
        [-2.4180,  8.7938,  2.2382, 59.7924]])


接着计算 softmax 权重，并查看某一行的分布：

In [2]:
weights = F.softmax(scores, dim=-1)  # 沿每一行做 softmax
print("Softmax 权重（第0行）:", weights[0])
# 很可能看到类似 [0.9999, 0, 0, 0]，梯度几乎为零

Softmax 权重（第0行）: tensor([1.0000e+00, 2.4295e-24, 3.2341e-26, 2.1091e-32])


**为什么这会失败？**
当 d 较大时，随机向量的点积方差约为 d。如果 q 和 k 的分量独立同分布，均值为 0，方差为 1，则 $\text{Var}(q \cdot k) = d$。点积值幅度随 d 增大而增大，经过 softmax 后几乎变成 one-hot，导致梯度消失。

**改进**
我们让点积除以 $\sqrt{d}$ 来将方差重新控制为 1：
$$
\text{Attention}(X) = \text{softmax}\left(\frac{X X^\top}{\sqrt{d}}\right) X
$$
这就是"缩放点积"的由来。此时 softmax 的输入不再过大，权重变得平缓。

In [3]:
d_k = d  # 此时键向量的维度
scaled_scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float))
weights_scaled = F.softmax(scaled_scores, dim=-1)
print("缩放后 Softmax 权重（第0行）:", weights_scaled[0])
# 现在权重分布更合理，例如 [0.2, 0.3, 0.1, 0.4]

缩放后 Softmax 权重（第0行）: tensor([9.9813e-01, 1.1153e-03, 6.4999e-04, 1.0957e-04])


这一步写完，我们手里就有了最简陋的"缩放点积自注意力"——每一行代码的目的：  
- `scores = X @ X.T`：获得所有词对之间的原始相似度。  
- `torch.sqrt(torch.tensor(d_k))`：计算缩放因子 $\sqrt{d_k}$。  
- `scaled_scores`：让点积的方差重新回到 1，防止 softmax 饱和。  
- `F.softmax(scaled_scores, dim=-1)`：将每行的相似度转化为概率分布（注意力权重）。  
- `weights_scaled @ X`（稍后会做）：用这些概率去取 X 中其他词的加权平均。

---

## 2. 引入可学习的 Q、K、V：让网络学会"查什么、键是什么、取什么"

**Idea**
上一步里，我们直接用 X 本身充当了"用来查询的词""被查询的键"以及"被聚合的值"。但模型需要灵活地学习不同的表示空间：  
- **查询 Q**：代表"我在找什么"。  
- **键 K**：代表"我含有什么信息"。  
- **值 V**：代表"如果你选中了我，我将提供什么内容"。  

为此，我们引入三个可训练的权重矩阵 $W_Q, W_K, W_V$，将原始输入 X 投影到不同角色。

**Mathematical expression**
给定 $X \in \mathbb{R}^{n \times d_{\text{model}}}$：
$$
Q = X W_Q \quad (W_Q \in \mathbb{R}^{d_{\text{model}} \times d_k})
$$
$$
K = X W_K \quad (W_K \in \mathbb{R}^{d_{\text{model}} \times d_k})
$$
$$
V = X W_V \quad (W_V \in \mathbb{R}^{d_{\text{model}} \times d_v})
$$
注意力计算变为：
$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$
注意 Q 和 K 的维度必须相同（都为 $d_k$），才能做内积；V 的维度 $d_v$ 可以是任意值，我们常取 $d_v = d_k$。

**Code & Output — 手动创建可训练权重并实现**
我们不用 `nn.Linear` 的一步到位，而是显式定义 $W_Q, W_K, W_V$ 矩阵，并追踪梯度。

In [4]:
torch.manual_seed(42)
n, d_model, d_k, d_v = 4, 64, 64, 64

X = torch.randn(n, d_model)          # [4, 64]

# 手动初始化权重矩阵，并设为需要梯度
W_Q = torch.randn(d_model, d_k, requires_grad=True)   # [64, 64]
W_K = torch.randn(d_model, d_k, requires_grad=True)
W_V = torch.randn(d_model, d_v, requires_grad=True)

# 计算 Q, K, V —— 每行都是 X 的线性变换
Q = X @ W_Q   # [4, 64]
K = X @ W_K   # [4, 64]
V = X @ W_V   # [4, 64]

print("Q 形状:", Q.shape)  # torch.Size([4, 64])

Q 形状: torch.Size([4, 64])


接下来计算缩放点积注意力，仍然要除以 $\sqrt{d_k}$：

In [5]:
# 计算注意力分数矩阵 (Q 与 K 转置相乘)
scores = Q @ K.T                     # [4, 4] — 每个查询与所有键的相似度
# 缩放
d_k_tensor = torch.tensor(d_k, dtype=torch.float)
scaled_scores = scores / torch.sqrt(d_k_tensor)  # 仍为 [4, 4]
# softmax 得到注意力权重
attn_weights = F.softmax(scaled_scores, dim=-1)   # 每行概率和为 1
print("注意力权重:\n", attn_weights)
# 加权聚合值向量
output = attn_weights @ V            # [4, 64]
print("注意力输出形状:", output.shape)  # [4, 64]

注意力权重:
 tensor([[0.0000e+00, 5.1177e-13, 1.0000e+00, 7.8474e-15],
        [4.4001e-43, 7.7893e-23, 1.0000e+00, 1.0418e-19],
        [4.1897e-16, 1.0000e+00, 2.8138e-18, 2.8741e-25],
        [1.0420e-02, 5.0713e-28, 9.8958e-01, 3.0125e-11]],
       grad_fn=<SoftmaxBackward0>)
注意力输出形状: torch.Size([4, 64])


**逐行解释：**  
- `W_Q, W_K, W_V` 是模型学习的参数，把输入映射到不同的空间。  
- `Q = X @ W_Q`：X 中每个词向量与 W_Q 相乘，得到对应的查询向量。  
- `K = X @ W_K`：得到键向量，用于被查询。  
- `V = X @ W_V`：得到值向量，最终的上下文信息将从 V 中抽取。  
- `scores = Q @ K.T`：每个查询与所有键计算点积，得到一个 n×n 的相关矩阵。  
- 除以 `sqrt(d_k)`：保证输入 softmax 的数值尺度稳定。  
- `F.softmax(..., dim=-1)`：沿键的方向归一化，得到每个查询对所有键的注意力分布。  
- `attn_weights @ V`：用注意力概率去取 V 中的信息，完成"软寻址"。

**此时我们完全拥有了 Transformer 中最核心的缩放点积注意力，并且它是可学习的。**

---

## 3. 多头：用多个"视角"同时关注不同位置

**Idea**
只用一个 Q/K/V 投影，模型可能会混合不同层面的信息（比如既想学语法又想学语义），导致注意力含糊。研究者的灵感是：并行做多组注意力，每组维度缩小，最后拼接。这就像 CNN 中多个卷积核提取不同特征。

**Mathematical expression**
设头数 h，每个头的维度 $d_k = d_v = d_{\text{model}} / h$。对每个头 i：
$$
\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)
$$
其中 $W_i^Q \in \mathbb{R}^{d_{\text{model}} \times d_k}$ 等。然后将所有头拼接：
$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W_O
$$
$W_O \in \mathbb{R}^{h d_v \times d_{\text{model}}}$ 将拼接的结果重新投影回原始维度。

**实现技巧**：把多个头的计算合并成一个批量矩阵乘法，而不是用 for 循环。我们将 Q、K、V 线性变换后重塑成 `[batch, n, h, d_k]`，然后转置为 `[batch, h, n, d_k]`，并行计算所有头的注意力。

**Code & Output**

In [6]:
torch.manual_seed(42)
n, d_model, h = 4, 64, 8        # 8 个头
d_k = d_model // h               # 每个头维度 8

X = torch.randn(n, d_model)

# 定义整体的 Q, K, V 投影矩阵（合并所有头的权重）
W_Q_all = torch.randn(d_model, h * d_k, requires_grad=True)   # [64, 64]
W_K_all = torch.randn(d_model, h * d_k, requires_grad=True)
W_V_all = torch.randn(d_model, h * d_k, requires_grad=True)

# 1. 投影并重塑
Q_all = X @ W_Q_all                      # [4, 64] -> [n, h*d_k]
# reshape 成 [n, h, d_k]，然后为了并行计算，交换维度为 [h, n, d_k]
Q = Q_all.view(n, h, d_k).transpose(0, 1)   # [8, 4, 8]
K = (X @ W_K_all).view(n, h, d_k).transpose(0, 1)  # 同上
V = (X @ W_V_all).view(n, h, d_k).transpose(0, 1)

# 2. 计算缩放点积注意力，在一个批次内对所有头进行
# Q: [h, n, d_k], K: [h, n, d_k] -> K 转置最后两维 [h, d_k, n]
scores = torch.matmul(Q, K.transpose(-2, -1))  # [h, n, n]
scaled_scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float))
attn_weights = F.softmax(scaled_scores, dim=-1) # [h, n, n]

# 3. 用注意力权重聚合 V
head_outputs = torch.matmul(attn_weights, V)   # [h, n, d_k]

# 4. 拼接头并做最终投影
# 先将 [h, n, d_k] 转回 [n, h, d_k] 然后变成 [n, h*d_k]
concat = head_outputs.transpose(0, 1).contiguous().view(n, h * d_k)  # [4, 64]

W_O = torch.randn(h * d_k, d_model, requires_grad=True)
multihead_output = concat @ W_O               # [4, 64]
print("多头注意力输出形状:", multihead_output.shape)  # torch.Size([4, 64])

多头注意力输出形状: torch.Size([4, 64])


**逐行要点：**  
- `W_Q_all` 的形状是 `[d_model, h*d_k]`，它其实等价于 h 个头各自的 $W_i^Q$ 横向拼接。  
- `.view(n, h, d_k).transpose(0,1)`：把"头"维度提到最前，这样后面 `matmul` 就能同时对 8 个头广播。  
- `scores = torch.matmul(Q, K.transpose(-2, -1))`：为每个头独立计算 n×n 注意力分数。  
- `softmax` 在最后一维进行，得到每个头内部的归一化权重。  
- `head_outputs.transpose(0,1).contiguous().view(...)` 把头拼回特征维度。  
- `W_O` 再投影，使输出的维度和输入 X 一致，方便堆叠下一个块。

---

## 4. 位置编码：告诉模型"第一个词"是什么意思

**Idea**
上面的注意力机制对词的顺序完全不敏感——无论输入顺序如何打乱，注意力加权求和的结果都一样。我们必须给词向量注入位置信息。  
研究者选择了正弦余弦位置编码，它不同位置的向量可以通过线性关系互相表达，有助于模型捕捉相对位置。

**Mathematical expression**
对位置 pos 和维度 i（从 0 到 d-1）：
$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)
$$
$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)
$$
然后将这个矩阵直接加到输入 X 上。

**Code & Output**

In [8]:
def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)            # 预分配
    position = torch.arange(0, max_len).unsqueeze(1).float()  # [max_len, 1]
    # 计算分母 10000^(2i/d)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                         (-torch.log(torch.tensor(10000.0)) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)  # 偶数位正弦
    pe[:, 1::2] = torch.cos(position * div_term)  # 奇数位余弦
    return pe

max_len, d_model = 4, 64
pe = positional_encoding(max_len, d_model)        # [4,64]
print("位置编码矩阵:\n", pe[:, :])  # 展示前8维
X_with_pos = X + pe           # 将位置编码逐元素加到词向量上
print("加入位置编码后 X 的形状:", X_with_pos.shape)  # [4, 64]

位置编码矩阵:
 tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  6.815

**每行解释**  
- `position` 生成列向量 [0,1,2,3]^T。  
- `div_term` 是 $1/10000^{2i/d}$ 的对数形式，避免数值计算不稳定。  
- 偶数索引用 sin，奇数索引用 cos，使每一维对应不同频率。  
- 最后直接加到 X，让模型知道每个词的绝对（及相对）位置。

---

## 5. 一个完整的 Transformer 块：注意力 + 前馈 + 残差 & 层归一化

**Idea**
将前面所有的部件组合成一个编码器块。研究者发现必须加入：
- 残差连接：让信息绕道传播，缓解梯度消失。
- 层归一化：稳定训练。
- 逐位置的前馈网络（FFN）：给每个位置的表示增加非线性变换，通常由两个线性层和激活函数构成。

**Mathematical expression（一个块）**
输入 $x$：
$$
\text{attn\_out} = \text{LayerNorm}(x + \text{MultiHead}(x))
$$
$$
\text{ffn\_out} = \text{LayerNorm}(\text{attn\_out} + \text{FFN}(\text{attn\_out}))
$$
FFN 为：
$$
\text{FFN}(z) = \text{ReLU}(z W_1 + b_1) W_2 + b_2
$$

**Code & Output**  
此处我们将前面写好的多头注意力封装为函数（不使用 `nn.MultiheadAttention`），并构建一个块。

In [9]:
# 手动实现多头注意力（加入位置编码后），用前面的方法
def multihead_attention(X, W_Q_all, W_K_all, W_V_all, W_O, h, d_k):
    n = X.size(0)
    Q = (X @ W_Q_all).view(n, h, d_k).transpose(0, 1)
    K = (X @ W_K_all).view(n, h, d_k).transpose(0, 1)
    V = (X @ W_V_all).view(n, h, d_k).transpose(0, 1)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float))
    attn_weights = F.softmax(scores, dim=-1)
    head_outs = torch.matmul(attn_weights, V)
    concat = head_outs.transpose(0,1).contiguous().view(n, h * d_k)
    return concat @ W_O

# 参数初始化
torch.manual_seed(42)
n, d_model, h = 4, 64, 8
d_k = d_model // h
X = torch.randn(n, d_model)
pe = positional_encoding(n, d_model)
X = X + pe

# 创建所有权重（可学习）
W_Q_all = torch.randn(d_model, h*d_k, requires_grad=True)
W_K_all = torch.randn(d_model, h*d_k, requires_grad=True)
W_V_all = torch.randn(d_model, h*d_k, requires_grad=True)
W_O     = torch.randn(h*d_k, d_model, requires_grad=True)

# --- 注意力子层 + 残差 & LayerNorm ---
attn_out = multihead_attention(X, W_Q_all, W_K_all, W_V_all, W_O, h, d_k)
# 残差连接并归一化（手动使用 nn.LayerNorm，这是基础的归一化工具）
layernorm1 = torch.nn.LayerNorm(d_model)
x1 = layernorm1(X + attn_out)   # [4,64]

# --- 前馈网络子层 + 残差 & LayerNorm ---
W1 = torch.randn(d_model, 4*d_model, requires_grad=True)   # FFN 通常中间维度扩大
b1 = torch.randn(4*d_model, requires_grad=True)
W2 = torch.randn(4*d_model, d_model, requires_grad=True)
b2 = torch.randn(d_model, requires_grad=True)

ffn_out = torch.relu(x1 @ W1 + b1) @ W2 + b2   # 逐位置前馈，依然是 [4,64]
layernorm2 = torch.nn.LayerNorm(d_model)
output = layernorm2(x1 + ffn_out)

print("Transformer 块最终输出形状:", output.shape)  # torch.Size([4, 64])
print("输出示例:\n", output)

Transformer 块最终输出形状: torch.Size([4, 64])
输出示例:
 tensor([[-2.5209e-01, -5.7204e-01, -1.1264e+00, -1.4590e+00, -1.9506e+00,
          9.3843e-01,  5.4356e-02,  2.1515e+00, -7.1827e-01, -8.4460e-02,
          4.0086e-01, -1.2247e+00,  4.9118e-01,  1.0764e+00,  2.9362e+00,
          1.1871e+00, -6.2128e-01,  5.5963e-01,  9.1302e-01,  2.5905e-03,
          1.4008e+00,  2.4822e-01, -1.0360e+00,  1.3724e-01,  5.0743e-01,
          8.0227e-02, -7.4759e-01, -7.8722e-01, -1.9114e-01,  3.2976e-01,
         -7.1438e-01,  8.9617e-02, -7.4960e-01,  1.7070e+00,  5.9082e-01,
         -1.1992e+00,  5.2099e-01, -2.2628e+00,  1.4264e+00,  3.2363e-01,
          4.8606e-01, -1.1213e-01, -2.2406e-01, -2.0287e+00,  1.4603e+00,
         -1.9108e-01, -7.4719e-01,  7.6749e-01, -1.0235e+00,  1.0515e+00,
          5.7543e-01,  5.3779e-01, -1.9119e-02, -3.0146e-01, -8.4053e-01,
         -6.1315e-01, -1.7799e+00,  2.7392e-01,  1.1506e+00,  5.8154e-01,
         -4.1657e-01, -5.9533e-01,  3.4123e-01, -7.0960e-01],
  

**说明：**  
- `X + attn_out` 是残差连接，把原始输入直接加到注意力结果上。  
- `LayerNorm` 对每个样本的 hidden 维度做归一化，稳定训练。  
- FFN 里的 `@ W1 + b1` 把每个位置独立地映射到更高维度，经过 ReLU 再压回原维度。  
- 以上所有操作均未使用 `nn.MultiheadAttention`，而是用明确的矩阵乘法重现。

---

## 回首：我们是如何一步步走到这里的？

我们经历的路径正对应了当年研究者的思维轨迹：  
1. **想要并行化并捕捉长距离依赖** → 直接算全序列内积相似度。  
2. **内积数值太大导致 softmax 梯度消失** → 除以 √d_k 缩放，使方差归一化。  
3. **希望网络学会"关注什么"** → 引入可学习的 Q, K, V 投影。  
4. **单一注意力混杂多种信息** → 切成多个头，从不同子空间聚合。  
5. **缺失序列顺序** → 加上正弦/余弦位置编码。  
6. **最终组件化** → 用残差连接、层归一化和前馈网络搭成可堆叠的块。

每一步都是先有 idea，然后落实到数学表达式，最后用最基础的张量操作（点积、softmax、reshape、转置）变成代码，并亲眼看到张量形状和数值是否符合预期。这正是当年发明 Transformer 的过程——没有现成的 `MultiheadAttention`，只有对注意力本质的追问与干净的矩阵计算。

---

## 6. 实例分析：多义词 "bank" 在 Transformer 中的处理

前面我们完整搭建了 Transformer 的核心组件。现在通过一个具体例子——
当遇到多义词 "bank" 时（"河岸"还是"银行"？），
看 Transformer 如何利用自注意力机制根据上下文动态调整词表示。

流程：
1. **6.1 初始状态**——随机词向量，没有任何语义
2. **6.2 训练词向量**——用 Skip-gram 让向量从共现中学习语义
3. **6.3 自注意力消歧义**——在具体句子中看 bank 如何在语境中被消歧义


### 6.1 初始状态：随机词向量

**6.1a  定义词表**

首先定义 9 个词。整个演示都基于这个小词表。
每个词将被映射到一个唯一的整数索引，方便张量操作。

In [49]:
import torch
import torch.nn.functional as F
torch.manual_seed(42)
#                                Lily  is  running  along  the  river  bank  money  rich
vocab = ['Lily', 'is', 'running', 'along', 'the', 'river', 'bank', 'money', 'rich']
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
print("词表:", vocab)


词表: ['Lily', 'is', 'running', 'along', 'the', 'river', 'bank', 'money', 'rich']


**6.1b  初始化嵌入矩阵**

每个词 $w$ 被表示为一个 $d$ 维实数向量 $\mathbf{e}_w \in \mathbb{R}^d$（此处 $d = 32$）。
初始时所有向量独立采样自标准正态分布 $\mathcal{N}(\mathbf{0}, \mathbf{I}_d)$。

注意初始化了两个矩阵：
- `embeddings`：输入嵌入，$[|\mathcal{V}|, d]$，每个词作为**目标词**时的表示
- `W_out`：输出嵌入矩阵，$[d, |\mathcal{V}|]$，每个词作为**上下文词**时的表示

In [50]:
d = 32
# embeddings[i] 是第 i 个词作为"目标词"时的向量
embeddings = torch.randn(len(vocab), d, requires_grad=True)
# W_out[:, j] 是第 j 个词作为"上下文词"时的向量
W_out = torch.randn(d, len(vocab), requires_grad=True)
print("embeddings 形状:", embeddings.shape)
print("W_out 形状:     ", W_out.shape)


embeddings 形状: torch.Size([9, 32])
W_out 形状:      torch.Size([32, 9])


**6.1c  余弦相似度**

衡量两个词 $v$ 和 $w$ 的语义相似度，使用**余弦相似度**：
$$
\text{cos}(\mathbf{e}_v, \mathbf{e}_w) = \frac{\mathbf{e}_v \cdot \mathbf{e}_w}{\|\mathbf{e}_v\| \cdot \|\mathbf{e}_w\|}
$$
值域 $[-1, 1]$：1 方向相同，0 正交（无关），-1 方向相反。

In [51]:
def cos_sim(v1, v2):
    """余弦相似度，值域 [-1, 1]"""
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()

print("自相似度 (cos(v, v)): %.4f (应为 1.0)" % cos_sim(torch.randn(d), torch.randn(d)))
print("随机向量对 (cos(v1, v2)): %.4f (应接近 0)" % cos_sim(torch.randn(d), torch.randn(d)))


自相似度 (cos(v, v)): -0.0365 (应为 1.0)
随机向量对 (cos(v1, v2)): -0.4664 (应接近 0)


**6.1d  观察初始向量值**

打印每个词的初始向量（前 16 维），以及 bank 与关键词的余弦相似度。
预期：所有相似度接近 0——**随机初始化没有语义关联**。

In [52]:
print("=" * 72)
print("6.1  初始随机词向量")
print("=" * 72)
print("词表大小:", len(vocab), "，向量维度: d =", d, "\n")

print("每个词的初始向量（前 16 维 / 共 32 维）:")
print("-" * 72)
for w in vocab:
    v = embeddings[word2idx[w]].detach().numpy()[:16]
    s = ', '.join(f'{x:7.3f}' for x in v)
    print("  %10s: [%s ...]" % (w, s))

bank_v = embeddings[word2idx['bank']].detach()
river_v = embeddings[word2idx['river']].detach()
money_v = embeddings[word2idx['money']].detach()

print()
print("初始余弦相似度:")
print("-" * 72)
print("  cos(bank, river)  = %.4f" % cos_sim(bank_v, river_v))
print("  cos(bank, money)  = %.4f" % cos_sim(bank_v, money_v))
print("  cos(river, money) = %.4f" % cos_sim(river_v, money_v))
print("  cos(river, running) = %.4f" % cos_sim(
    river_v, embeddings[word2idx['running']].detach()))


6.1  初始随机词向量
词表大小: 9 ，向量维度: d = 32 

每个词的初始向量（前 16 维 / 共 32 维）:
------------------------------------------------------------------------
        Lily: [  1.927,   1.487,   0.901,  -2.106,   0.678,  -1.235,  -0.043,  -1.605,  -0.752,   1.649,  -0.392,  -1.404,  -0.728,  -0.559,  -0.769,   0.762 ...]
          is: [ -1.385,  -0.871,  -0.223,   1.717,   0.319,  -0.425,   0.306,  -0.775,  -1.558,   0.996,  -0.880,  -0.601,  -1.274,   2.123,  -1.235,  -0.488 ...]
     running: [  1.445,   0.856,   2.218,   0.523,   0.347,  -0.197,  -1.055,   1.278,  -0.172,   0.524,   0.057,   0.426,   0.575,  -0.642,  -2.206,  -0.751 ...]
       along: [ -2.510,   0.488,   0.785,   0.029,   0.641,   0.583,   1.067,  -0.450,  -0.185,   0.753,   0.405,   0.178,   0.265,   1.273,  -0.001,  -0.304 ...]
         the: [  1.931,   1.012,  -1.436,  -1.130,  -0.136,   1.635,   0.655,   0.576,   1.142,   0.019,  -1.806,   0.925,  -0.375,   1.033,  -0.687,   0.637 ...]
       river: [ -1.901,   0.229,   0.025,  -0.34

**6.1e  随机初始化分布验证**

独立同分布高维随机向量的余弦相似度服从 $\mathcal{N}(0, 1/d)$。
$d=32$ 时理论标准差 $1/\sqrt{32}\approx 0.177$。
下面统计所有 36 个词对的余弦相似度，验证其均值接近 0、标准差接近理论值。

In [53]:
import itertools
all_vecs = embeddings.detach()
sim_list = []
for w1, w2 in itertools.combinations(list(vocab), 2):
    v1 = all_vecs[word2idx[w1]]
    v2 = all_vecs[word2idx[w2]]
    sim_list.append(cos_sim(v1, v2))

sim_t = torch.tensor(sim_list)
print("=" * 72)
print("独立性验证：随机初始化向量的余弦相似度统计")
print("=" * 72)
print("  词对总数:", len(sim_list))
print("  平均相似度: %.4f" % sim_t.mean().item())
print("  标准差:     %.4f" % sim_t.std().item())
print("  最大值:     %.4f" % sim_t.max().item())
print("  最小值:     %.4f" % sim_t.min().item())
print()
print("  理论 N(0, 1/d) 标准差 %.4f" % (1 / d ** 0.5))
print()
print("结论: 平均相似度接近 0，词向量之间确实相互独立，无语义关联。" if abs(sim_t.mean().item()) < 0.1 else "注意: 平均相似度偏离 0 较多。")


独立性验证：随机初始化向量的余弦相似度统计
  词对总数: 36
  平均相似度: 0.0431
  标准差:     0.1605
  最大值:     0.3571
  最小值:     -0.2932

  理论 N(0, 1/d) 标准差 0.1768

结论: 平均相似度接近 0，词向量之间确实相互独立，无语义关联。


**6.1f  初始状态总结**

以上验证了：
- 9 个词各自对应一个 32 维的随机向量
- 所有词对的余弦相似度均值 ~0.04，标准差接近 $1/\sqrt{32}$
- bank 与 river、money 等的相似度都接近 0，**没有任何语义关联**

接下来通过训练让词向量学习语义。

### 6.2 训练词向量：Skip-gram 模型

**6.2a  定义训练语料**

设计 6 个短句。关键设计：
- bank 出现在两种不同语境中：与 river 共现（河岸）| 与 money 共现（银行）
- money 还与 rich 共现，且**不含 bank**，给 money 一个独立语义锚点
- 这样 $W_{\text{out}}[:, \text{money}]$ 有独特预测词 {rich}，$W_{\text{out}}[:, \text{river}]$ 有独特预测词 {along}，
  避免两者因共享 {the, bank} 作为唯一预测词而过于相似

In [54]:
sentences = [
    ['the', 'river', 'bank'],                     # bank = 河岸
    ['running', 'along', 'the', 'river', 'bank'], # bank = 河岸
    ['the', 'money', 'bank'],                     # bank = 银行
    ['the', 'rich', 'money'],                     # money + rich（无 bank）
    ['Lily', 'is', 'running', 'along', 'the', 'river'],
    ['Lily', 'is', 'running'],
]
print("训练语料（共 6 个短句）:")
for s in sentences:
    print("  " + " ".join(s))


训练语料（共 6 个短句）:
  the river bank
  running along the river bank
  the money bank
  the rich money
  Lily is running along the river
  Lily is running


**6.2b  构造 Skip-gram 训练样本**

给定目标词，预测它周围的上下文词（窗口=2）。
例如 "the river bank" 生成 (the→river), (the→bank), (river→the), (river→bank), (bank→the), (bank→river)。

In [55]:
window = 2
train_pairs = []  # (目标词索引, 上下文词索引)
for sent in sentences:
    for i, tgt in enumerate(sent):
        if tgt not in word2idx: continue
        for j in range(max(0, i - window), min(len(sent), i + window + 1)):
            if i != j and sent[j] in word2idx:
                train_pairs.append((word2idx[tgt], word2idx[sent[j]]))

print("Skip-gram 训练样本数:", len(train_pairs))
print("前 6 个样本:")
for tgt, ctx in train_pairs[:6]:
    print("  (%s -> %s)" % (idx2word[tgt], idx2word[ctx]))
print("  ...")


Skip-gram 训练样本数: 56
前 6 个样本:
  (the -> river)
  (the -> bank)
  (river -> the)
  (river -> bank)
  (bank -> the)
  (bank -> river)
  ...


**6.2c  损失函数与训练循环**

对每个训练对 $(w_{\text{tgt}}, w_{\text{ctx}})$：
$$
\mathbf{s} = \mathbf{e}_{\text{tgt}}^\top W_{\text{out}},
\quad
P(w_{\text{ctx}} \mid w_{\text{tgt}}) = \frac{\exp(s_{w_{\text{ctx}}})}{\sum_k \exp(s_k)},
\quad
\mathcal{L} = -\log P(w_{\text{ctx}} \mid w_{\text{tgt}})
$$
每 30 轮打印一次关键相似度和向量值，观察它们从随机值逐渐变化。

In [56]:
optimizer = torch.optim.SGD([embeddings, W_out], lr=0.3)
print("训练过程（每 30 轮打印一次）:")
print("=" * 72)
for epoch in range(150):
    total_loss = 0.0
    for tgt, ctx in train_pairs:
        logits = embeddings[tgt] @ W_out           # [vocab_size]
        loss = F.cross_entropy(logits.unsqueeze(0), torch.tensor([ctx]))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 30 == 0:
        out = W_out.T.detach()
        bv = out[word2idx['bank']]
        rv = out[word2idx['river']]
        mv = out[word2idx['money']]
        print()
        print("  Epoch %3d, Loss = %.6f" % (epoch, total_loss / len(train_pairs)))
        print("    cos(bank, river)  = %.4f" % cos_sim(bv, rv))
        print("    cos(bank, money)  = %.4f" % cos_sim(bv, mv))
        for w in ['bank', 'river', 'money']:
            vec = out[word2idx[w]].numpy()[:8]
            s = ', '.join('%7.3f' % x for x in vec)
            print("    %8s: [%s ...]" % (w, s))

print()
print("训练完成。")


训练过程（每 30 轮打印一次）:

  Epoch   0, Loss = 10.089909
    cos(bank, river)  = 0.4942
    cos(bank, money)  = 0.0552
        bank: [ -0.300,   0.733,  -0.412,  -0.469,  -0.848,   0.205,   0.569,  -0.544 ...]
       river: [ -0.208,   0.880,  -0.668,   0.440,   0.649,   0.823,   1.000,  -0.305 ...]
       money: [ -0.008,   0.231,  -2.487,   0.052,  -1.408,   0.632,  -0.358,  -0.105 ...]

  Epoch  30, Loss = 2.002591
    cos(bank, river)  = 0.6179
    cos(bank, money)  = 0.6490
        bank: [  0.051,   0.337,  -0.550,  -0.058,   0.404,   0.189,   0.312,  -0.688 ...]
       river: [  0.246,   0.100,  -0.726,  -0.107,   0.032,   0.301,   0.194,   0.131 ...]
       money: [ -0.250,   0.306,  -1.094,   0.233,  -0.200,   0.610,   0.428,  -0.460 ...]

  Epoch  60, Loss = 2.001420
    cos(bank, river)  = 0.6334
    cos(bank, money)  = 0.4674
        bank: [  0.139,   0.134,  -0.540,  -0.189,  -0.150,   0.270,   0.343,   0.408 ...]
       river: [ -0.220,   0.412,  -0.755,  -0.052,  -0.320,   0.404,

**6.2d  训练后评估**

使用 $W_{\text{out}}$ 的列向量作为词表示。输出嵌入编码了「哪些目标词会预测我作为上下文」，比输入嵌入更能反映共现结构。

- `cos(bank, river)` 较高（~0.7）：共现于 river bank 语境
- `cos(bank, money)` 也较高（~0.6）：共现于 money bank 语境
- `cos(bank, rich)` 也较高（~0.6）：rich 通过 money 与 bank 间接关联
- **bank 的静态向量被 river 和 money 同时拉近 → 语义折中**

In [57]:
out = W_out.T.detach()
bi, ri, mi = word2idx['bank'], word2idx['river'], word2idx['money']
rni, li = word2idx['running'], word2idx['Lily']
rii = word2idx['rich']

print("=" * 72)
print("训练后评估（使用 W_out 列向量）:")
print("=" * 72)
print("  cos(bank, river)  = %.4f  (共现 river bank)" % cos_sim(out[bi], out[ri]))
print("  cos(bank, money)  = %.4f  (共现 money bank)" % cos_sim(out[bi], out[mi]))
print("  cos(bank, rich)   = %.4f  (经 money 间接关联)" % cos_sim(out[bi], out[rii]))
print("  cos(river, money) = %.4f" % cos_sim(out[ri], out[mi]))
print("  cos(river, running)= %.4f  (共现沿河跑)" % cos_sim(out[ri], out[rni]))
print("  cos(money, rich)  = %.4f  (共现 rich money)" % cos_sim(out[mi], out[rii]))
print()
print("注意: bank 与 river 的相似度应略高于 bank 与 money，") 
print("因为 river 有更多独有共现词 (along, running, Lily, is)，") 
print("而 money 的独有共现词 (rich) 也与 bank 间接关联。")


训练后评估（使用 W_out 列向量）:
  cos(bank, river)  = 0.7084  (共现 river bank)
  cos(bank, money)  = 0.6251  (共现 money bank)
  cos(bank, rich)   = 0.6929  (经 money 间接关联)
  cos(river, money) = 0.6511
  cos(river, running)= 0.6428  (共现沿河跑)
  cos(money, rich)  = 0.5663  (共现 rich money)

注意: bank 与 river 的相似度应略高于 bank 与 money，
因为 river 有更多独有共现词 (along, running, Lily, is)，
而 money 的独有共现词 (rich) 也与 bank 间接关联。


### 6.3 自注意力：在语境中消歧义

**6.3a  加载输出嵌入与句子**

用训练好的 $W_{\text{out}}^\top$ 作为词向量。句子中的 bank 将接受自注意力，更关注 river，向"河岸"偏移。

In [58]:
word_vecs = W_out.T.detach()
sentence = ['Lily', 'is', 'running', 'along', 'the', 'river', 'bank']
print("待处理句子:", " ".join(sentence))
idxs = torch.tensor([word2idx[w] for w in sentence])
X = word_vecs[idxs]


待处理句子: Lily is running along the river bank


**6.3b  计算自注意力权重**

$$
A = \text{softmax}\left( \frac{O_{\text{norm}} O_{\text{norm}}^\top}{\tau} \right),
\quad \tau = 0.3
$$
步骤：1) L2 归一化 2) 余弦相似度矩阵 3) 缩放 + softmax

In [59]:
X_norm = X / X.norm(dim=1, keepdim=True)
cos_scores = X_norm @ X_norm.T
temperature = 0.3
attn_weights = F.softmax(cos_scores / temperature, dim=-1)

print("自注意力权重矩阵（行 = 查询词，列 = 被关注词）:")
print("-" * 90)
header = "          " + "".join("%9s" % w for w in sentence)
print(header); print("-" * 90)
for i, w in enumerate(sentence):
    row = "%12s" % w
    for j in range(len(sentence)):
        row += "%8.3f " % attn_weights[i, j].item()
    print(row)


自注意力权重矩阵（行 = 查询词，列 = 被关注词）:
------------------------------------------------------------------------------------------
               Lily       is  running    along      the    river     bank
------------------------------------------------------------------------------------------
        Lily   0.415    0.163    0.080    0.183    0.070    0.037    0.052 
          is   0.156    0.398    0.137    0.087    0.114    0.069    0.040 
     running   0.087    0.154    0.449    0.091    0.038    0.137    0.044 
       along   0.156    0.077    0.072    0.355    0.100    0.074    0.166 
         the   0.063    0.105    0.031    0.105    0.369    0.145    0.182 
       river   0.035    0.068    0.120    0.082    0.154    0.393    0.149 
        bank   0.047    0.037    0.037    0.176    0.185    0.142    0.376 


**6.3c  分析 bank 的注意力分配**

看 bank（第 6 行）对各词的注意力：river 获得了多少权重？与 bank-river 高相似度一致吗？

In [60]:
print("'bank' 的注意力分配（按权重降序）:")
print("-" * 48)
bank_attn = attn_weights[6]
for rank, idx in enumerate(torch.argsort(bank_attn, descending=True)):
    word = sentence[idx]
    weight = bank_attn[idx].item()
    bar = chr(9608) * int(weight * 100)
    note = " (自我)" if idx == 6 else ""
    print("  #%d -> %10s: %.4f %s%s" % (rank+1, word, weight, bar, note))


'bank' 的注意力分配（按权重降序）:
------------------------------------------------
  #1 ->       bank: 0.3760 █████████████████████████████████████ (自我)
  #2 ->        the: 0.1849 ██████████████████
  #3 ->      along: 0.1758 █████████████████
  #4 ->      river: 0.1423 ██████████████
  #5 ->       Lily: 0.0469 ████
  #6 ->         is: 0.0374 ███
  #7 ->    running: 0.0368 ███


**6.3d  计算语境化表示**
$$
\mathbf{o}'_i = \sum_j A_{i,j} \, \mathbf{o}_j
$$

In [61]:
contextualized = attn_weights @ X
bank_orig = X[6]; bank_new = contextualized[6]
print("'bank' 原始向量 vs 语境化向量:")
print("-" * 72)
print("  原始（前12维）: [%s ...]" %
      ', '.join('%7.3f' % x for x in bank_orig.numpy()[:12]))
print("  语境化（前12维）: [%s ...]" %
      ', '.join('%7.3f' % x for x in bank_new.numpy()[:12]))


'bank' 原始向量 vs 语境化向量:
------------------------------------------------------------------------
  原始（前12维）: [ -0.301,   0.482,  -0.638,   0.024,  -0.163,   0.389,   0.524,  -0.291,   0.112,   0.780,   0.111,  -0.416 ...]
  语境化（前12维）: [ -0.163,   0.505,  -0.482,  -0.141,   0.033,   0.222,   0.411,  -0.397,   0.237,   0.682,  -0.018,  -0.376 ...]


**6.3e  相似度变化分析**

对比 bank 语境化前后与各词的余弦相似度变化：
- `bank vs river`（在当前句中）：期待**明显上升**（吸收河岸语义）
- `bank vs money`（不在当前句）：期待**持平或下降**（不应被拉近）
- `bank vs rich`（不在当前句）：期待**下降**（money 的关联词也不在句中）

如果 bank-river 上升而 bank-money/rich 下降或持平，证明自注意力成功针对当前语境消歧义，而非简单地拉向所有词的平均。

In [62]:
river_vec = X[5]
money_vec = word_vecs[word2idx['money']]
rich_vec = word_vecs[word2idx['rich']]

print("  相似度变化：")
print("  %-35s %8s %8s %10s" % ("", "原始", "语境化后", "变化"))
print("  " + "-" * 65)
pairs = [
    ("bank vs river  ", bank_orig, river_vec, bank_new, river_vec),
    ("bank vs money  ", bank_orig, money_vec, bank_new, money_vec),
    ("bank vs rich   ", bank_orig, rich_vec, bank_new, rich_vec),
]
for label, v1, v2, v1n, v2n in pairs:
    o = cos_sim(v1, v2); n = cos_sim(v1n, v2n)
    d = n - o; sgn = "+" if d > 0 else ""
    print("  cos(%s) = %.4f -> %.4f  (%s%.4f)" % (label, o, n, sgn, d))
print()
print("关键: bank vs river 应明显上升，bank vs money/rich 应持平或下降。")
print("这证明自注意力基于上下文有针对性地调整表示。")


  相似度变化：
                                            原始     语境化后         变化
  -----------------------------------------------------------------
  cos(bank vs river  ) = 0.7084 -> 0.8140  (+0.1056)
  cos(bank vs money  ) = 0.6251 -> 0.5681  (-0.0570)
  cos(bank vs rich   ) = 0.6929 -> 0.5367  (-0.1562)

关键: bank vs river 应明显上升，bank vs money/rich 应持平或下降。
这证明自注意力基于上下文有针对性地调整表示。


**6.3f  结论**

1. **静态词向量的困境**：bank 与 river 和 money 都有较高相似度，因为它同时出现在两种语境中，被拉向两个方向，形成"语义折中"。

2. **自注意力如何解决**：在具体句子中，bank 通过自注意力看到所有其他词。river 与 bank 的表示高度相关（训练使它们靠近），因此 river 获得较高注意力权重，bank 从中吸收「河岸」语义。

3. **量化效果**：bank-river 相似度明显上升（~+0.1），而当前句子中未出现的 money/rich 持平或下降。

4. **扩展**：若换到 "the money bank"，bank 会更多关注 money，消歧义为"银行"。

5. **现实 Transformer**：本演示用余弦相似度简化模拟注意力；真实模型通过可学习的 Q/K/V 投影矩阵能学出更精细的模式。

> 核心启示：自注意力让词表示从「孤立查找表」升级为「动态上下文组装」。

---

## 附录：极小词表下的词向量训练——设计决策与局限

### 1. 遇到的现象

在最初的设计中，词表包含 `financial` 一词，训练语料为：
```
['the', 'river', 'bank']        # bank = 河岸
['running', 'along', 'the', 'river', 'bank']
['the', 'money', 'bank']        # bank = 银行
['the', 'financial', 'bank']    # bank = 银行（金融语境）
['Lily', 'is', 'running', 'along', 'the', 'river']
['Lily', 'is', 'running']
```

训练后评估发现 `cos(bank, money) > cos(bank, river)`，更严重的是在自注意力语境化后，
bank 与 money 的相似度**非但不降反而继续上升**（约 +0.03）：
```
cos(bank vs river)  = 0.59 -> 0.72  (+0.13)  ← 预期
cos(bank vs money)  = 0.69 -> 0.72  (+0.03)  ← 异常：不应拉近不在句中的 money
```

### 2. 根因诊断

通过分析 Skip-gram 中 $W_{\text{out}}$ 的预测词集合，发现：

| 输出嵌入 | 被哪些目标词的嵌入预测？ |
|---------|------------------------|
| $W_{\text{out}}[:, \text{river}]$ | {**the**, **bank**, along, running, Lily, is} |
| $W_{\text{out}}[:, \text{money}]$ | {**the**, **bank**} |
| $W_{\text{out}}[:, \text{financial}]$ | {**the**, **bank**} |

money 的唯一预测词 {the, bank} 与 river 的预测词完全重叠，且 financial 也与 bank 紧密关联（`financial bank`），
导致 money 和 financial 的输出嵌入都与 bank 高度相似。bank 的向量在两种语境间摇摆，
自注意力在句子中拉近 river 的同时也因向量空间的整体偏移而捎带了 money。

### 3. 修复方案

将 `financial` 替换为 `rich`，同时调整训练句子使 money 拥有不同于 river 的独立语境：
```
['the', 'rich', 'money']    # money 与 rich 共现，不含 bank
```

修改后的预测词集合：

| 输出嵌入 | 被哪些目标词的嵌入预测？ |
|---------|------------------------|
| $W_{\text{out}}[:, \text{river}]$ | {the, bank, **along**, running, Lily, is} |
| $W_{\text{out}}[:, \text{money}]$ | {the, bank, **rich**} |

现在两人各有独特预测词：river 有 {along}，money 有 {rich}。输出嵌入不再完全重叠。

修复后的结果：
```
cos(bank vs river)  = 0.71 -> 0.81  (+0.11)  ← 上升
cos(bank vs money)  = 0.63 -> 0.57  (-0.06)  ← 下降 ✓
cos(bank vs rich)   = 0.69 -> 0.54  (-0.16)  ← 下降 ✓
```

bank 的静态向量本身就与 river 更近（0.71 > 0.63），语境化后只向 river 靠近、远离 money/rich——消歧义行为变得干净明确。

### 4. 局限性讨论

#### 4.1 极小词表下的"每个词都很珍贵"

本演示的词表仅 9 个词。在这种极稀疏的设置下，**每个词在训练语料中的出现模式**直接影响所有其他词的嵌入质量：

- 如果 money 只出现在 {the, bank} 语境中，它与 bank 的绑定过于紧密，无法形成独立的语义表示
- 如果 river 的语境过于单一，它与 bank 的共现也无法充分体现在嵌入空间中
- 选择一个"桥梁词"（如 financial）可能无意中拉近两个本应分离的语义群

在真实大规模训练中，每个词有海量的上下文示例，个别缺失不会造成问题。但在教学演示中，
**词表设计本身就是模型设计的一部分**——它决定了哪些语义关系可以被成功编码。

#### 4.2 Skip-gram 在小语料上的行为

Skip-gram 依赖大量上下文样本来学习有意义的嵌入。当语料极小时：
- 损失函数快速收敛到接近随机水平的平台区（本演示中 Loss 停在 ~2.00）
- 嵌入之间的相似度主要由**共享预测词的个数和频率**决定，而非真正的语义距离
- `W_{\text{out}}$ 的列向量在某些方向上被少数几个目标词过度拉伸

#### 4.3 随机初始化与结果稳定性

本演示固定了随机种子 `torch.manual_seed(42)`。如果更换种子，初始嵌入不同，训练后的相似度数值会有变化，
但定性趋势（bank-river > bank-money）在修复后的语料下保持稳定。
若使用不同的种子，上述修复的数值（如 0.71、0.63 等）会变化，但 river 上升、money 下降的方向性结论不变。

#### 4.4 自注意力的简化

本演示使用余弦相似度 + 温度缩放代替可学习的 $W_Q, W_K, W_V$ 投影。这意味着：
- 注意力权重完全由嵌入空间中的余弦距离决定，无法重新缩放或旋转
- 真实 Transformer 的 Q/K 投影可以学出"关注什么"的策略，即使静态嵌入有噪声也能补偿
- 因此，真实模型对词表选择的敏感度低于本简化演示

### 5. 实践启示

1. **在小规模原型中，词表设计是模型设计**——删除或替换一个词可能彻底改变结果
2. **诊断工具**：分析 $W_{\text{out}}$ 的预测词集合可以快速定位嵌入过度相关的原因
3. **对比指标**：不要只看静态相似度，更要看**语境化前后的变化方向**——它反映了注意力是否在按预期工作
4. **教学简化与现实差距**：余弦注意力简化了真实 Transformer 的 Q/K/V 机制，
   真实场景中可学习的投影矩阵能自适应地补偿数据中的偏差

---

*本附录记录了第 6 节在设计与调试过程中的一次关键迭代，展示了在小样本条件下进行词向量训练的敏感性分析。*